# Desafio Sobrevivendo ao Titanic (Parte 2) — Superando o Limite Linear com `MLP`

Na Parte 1 treinamos um único neurônio linear (**Perceptron**). Agora vamos evoluir para um **Perceptron Multicamadas (PMC / MLP)** capaz de combinar variáveis de forma não linear (capturando interações como *"mulheres da 1ª/2ª classe"* vs. *"homens da 3ª classe"*).

## O Desafio (Parte 2 — Rede Multicamadas)
1. Prepare os dados de `../arquivos-csv/titanic_train.csv` e `../arquivos-csv/titanic_exercise.csv` sem vazamento de dados.
2. Treine um **MLP** (usando `MLPClassifier` do `Scikit-Learn` ou implementando a classe `MLP` em `NumPy` puro com camada oculta, *bias* e *Backpropagation*).
3. Compare a acurácia de validação com o Perceptron da Parte 1 e gere o arquivo `.csv` final!

In [ ]:
# Escreva sua implementação de MLP aqui!

<details>
<summary><b>Clique aqui para ver o Gabarito 2.1 — MLP com Scikit-Learn (Comentado)</b></summary>

```python
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

train_df = pd.read_csv("../arquivos-csv/titanic_train.csv", na_values=["?"])
test_df = pd.read_csv("../arquivos-csv/titanic_exercise.csv", na_values=["?"])

def preparar_dados(df_treino, df_teste):
    def extrair(df):
        X = pd.DataFrame()
        X["pclass"] = pd.to_numeric(df["pclass"], errors="coerce")
        X["is_male"] = df["sex"].map({"male": 1, "female": 0})
        X["age"] = pd.to_numeric(df["age"], errors="coerce")
        X["family"] = pd.to_numeric(df["sibsp"], errors="coerce") + pd.to_numeric(df["parch"], errors="coerce") + 1
        X["fare"] = pd.to_numeric(df["fare"], errors="coerce")
        titulos = df["name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
        X["title"] = titulos.map({"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4}).fillna(5)
        return X

    X_tr, X_te = extrair(df_treino), extrair(df_teste)
    med_titulo = X_tr.groupby("title")["age"].median()
    med_geral = X_tr.median()

    for tab in (X_tr, X_te):
        tab["age"] = tab["age"].fillna(tab["title"].map(med_titulo))
        tab.fillna(med_geral, inplace=True)
    return X_tr, X_te

X_all, X_ex = preparar_dados(train_df, test_df)
y_all = train_df["survived"].to_numpy()

# 1. Medindo ganho real em conjunto de validação (20%)
X_tr, X_val, y_tr, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
sc_val = StandardScaler()
# Arquitetura com 2 camadas ocultas (32 e 16 neurônios) e regularização L2 (alpha=0.05) contra overfitting
mlp_val = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.05, max_iter=1500, random_state=42)
mlp_val.fit(sc_val.fit_transform(X_tr), y_tr)
acc_mlp = accuracy_score(y_val, mlp_val.predict(sc_val.transform(X_val))) * 100.0
print(f"Acurácia de validação do MLP (Scikit-Learn): {acc_mlp:.2f}%")

# 2. Treinando no conjunto completo e salvando solução
sc_final = StandardScaler()
X_all_sc = sc_final.fit_transform(X_all)
X_ex_sc = sc_final.transform(X_ex)

mlp_final = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu", alpha=0.05, max_iter=1500, random_state=42)
mlp_final.fit(X_all_sc, y_all)

y_pred = mlp_final.predict(X_ex_sc)
pd.DataFrame({"id": test_df["id"], "survived": y_pred}).to_csv("solution_mlp_sklearn.csv", index=False)
print("Arquivo gerado com sucesso: solution_mlp_sklearn.csv")
```

</details>

<details>
<summary><b>Clique aqui para ver o Gabarito 2.2 — MLP com Backpropagation em Python/NumPy Puro (Comentado)</b></summary>

```python
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

train_df = pd.read_csv("../arquivos-csv/titanic_train.csv", na_values=["?"])
test_df = pd.read_csv("../arquivos-csv/titanic_exercise.csv", na_values=["?"])

def preparar_escalonado(df_treino, df_teste):
    def extrair(df):
        X = pd.DataFrame()
        X["pclass"] = pd.to_numeric(df["pclass"], errors="coerce")
        X["is_male"] = df["sex"].map({"male": 1, "female": 0})
        X["age"] = pd.to_numeric(df["age"], errors="coerce")
        X["family"] = pd.to_numeric(df["sibsp"], errors="coerce") + pd.to_numeric(df["parch"], errors="coerce") + 1
        X["fare"] = pd.to_numeric(df["fare"], errors="coerce")
        titulos = df["name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
        X["title"] = titulos.map({"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4}).fillna(5)
        return X

    X_tr, X_te = extrair(df_treino), extrair(df_teste)
    med_titulo = X_tr.groupby("title")["age"].median()
    med_geral = X_tr.median()
    for tab in (X_tr, X_te):
        tab["age"] = tab["age"].fillna(tab["title"].map(med_titulo))
        tab.fillna(med_geral, inplace=True)

    scaler = StandardScaler()
    return scaler.fit_transform(X_tr), scaler.transform(X_te)

X_train_sc, X_test_sc = preparar_escalonado(train_df, test_df)
y_train = train_df["survived"].to_numpy()

class MLPTitanicPuro:
    '''Rede Neural 1 camada oculta com vetores de bias explícitos e gradiente médio estável.'''
    def __init__(self, n_in, n_hid, n_out=1, taxa_aprendizado=0.5, seed=42):
        rng = np.random.default_rng(seed)
        # Pesos pequenos evitam saturação inicial da função sigmóide
        self.W1 = rng.normal(0.0, 0.3, size=(n_in, n_hid))
        self.b1 = np.zeros((1, n_hid))   # Vetor de viés (bias) da camada oculta
        self.W2 = rng.normal(0.0, 0.3, size=(n_hid, n_out))
        self.b2 = np.zeros((1, n_out))   # Vetor de viés (bias) da camada de saída
        self.lr = taxa_aprendizado

    @staticmethod
    def _sigmoide(u):
        # np.clip protege contra overflow numérico em exp(-u)
        return 1.0 / (1.0 + np.exp(-np.clip(u, -50.0, 50.0)))

    def treinar(self, X, y, epocas=1500):
        N = X.shape[0]
        y_col = y.reshape(-1, 1)  # Coluna (N x 1)

        for _ in range(epocas):
            # 1. Forward Pass em toda a matriz X (N x n_in) de uma só vez!
            h = self._sigmoide(X @ self.W1 + self.b1)   # Saída oculta (N x n_hid)
            o = self._sigmoide(h @ self.W2 + self.b2)   # Saída final (N x 1)

            # 2. Backward Pass (Regra da Cadeia)
            delta_o = (y_col - o) * (o * (1.0 - o))     # Gradiente local na saída
            delta_h = (delta_o @ self.W2.T) * (h * (1.0 - h))  # Gradiente na oculta

            # 3. Atualização pela MÉDIA do lote (divisão por N evita passos gigantescos!)
            self.W2 += (self.lr / N) * (h.T @ delta_o)
            self.b2 += (self.lr / N) * np.sum(delta_o, axis=0, keepdims=True)
            self.W1 += (self.lr / N) * (X.T @ delta_h)
            self.b1 += (self.lr / N) * np.sum(delta_h, axis=0, keepdims=True)

    def prever(self, X):
        h = self._sigmoide(X @ self.W1 + self.b1)
        o = self._sigmoide(h @ self.W2 + self.b2)
        return (o.ravel() >= 0.5).astype(int)

# Treina o MLP Puro com 16 neurônios ocultos e gera o CSV
rede = MLPTitanicPuro(n_in=X_train_sc.shape[1], n_hid=16, taxa_aprendizado=0.8, seed=42)
rede.treinar(X_train_sc, y_train, epocas=2000)

y_pred_puro = rede.prever(X_test_sc)
pd.DataFrame({"id": test_df["id"], "survived": y_pred_puro}).to_csv("solution_mlp_puro.csv", index=False)
print("Arquivo gerado com sucesso: solution_mlp_puro.csv")
```

</details>